In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain.chains import create_history_aware retriever
from langchain_core.output_parsers import StrOutputParser

In [6]:
loaded_text=TextLoader("self.txt",encoding="utf-8")
docs=loaded_text.load()

text_splitters=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n","\n"," ",""]
)

chunks=text_splitters.split_documents(docs)


In [7]:
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

## FAISS store

In [9]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
# vectorstore.save_local("faiss_index")

# loaded_vectorstore=FAISS.load_local(
#     "faiss_index",
#     embeddings,
#     allow_dangerous_deserialisation=True
# )

In [ ]:
llm=ChatOpenAI(
    model="gpt-3.5-turbo"
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever=vectorstore.as_retriever(k=3)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
simple_prompt="""
 ("system","You are a helpful assistant, and your work is to help the user with following context \n Context: {context}. If
 you don't know the answer simply say you don't know "),
 ("human",{question})
"""

rag_chain=(
    {
        "context":retriever|format_docs,
        "question":RunnablePassthrough()
    }|
    simple_prompt
    |llm
    |StrOutputParser()
)
